# Quantifizierung des gemeinsamen Design-Erbes einer Leistungstransformator-Flotte mit PROC INBREED

## Zusammenfassung

Die Verteiltransformatoren eines Netzbetreibers werden in aufeinanderfolgenden Design-Generationen entwickelt, wobei jedes neue Modell von zwei Vorgänger-Designs abstammt. Dieses Notebook behandelt diese technische Genealogie als Pedigree und nutzt **PROC INBREED**, um Inzucht- und additive Verwandtschafts- (Kovarianz-) Koeffizienten zu berechnen und so zu quantifizieren, wie viel Design-Erbe zwei beliebige Anlagen teilen.

Das Pedigree ist so aufgebaut, dass die Struktur interpretierbar ist: Zwei der vier aktuellen Flottenmodelle stammen von einem Paar **voller Geschwister**-Vorgängerdesigns ab und tragen daher ein konzentriertes Erbe, während die anderen auf disjunkten Abstammungslinien beruhen. Die Prozedur ermittelt dies exakt. Die beiden von Geschwistern abstammenden Modelle (`G3_T1`, `G3_T3`) tragen jeweils einen Inzuchtkoeffizienten von **F = 0.25**; die anderen beiden (`G3_T2`, `G3_T4`) liegen bei **F = 0**. Der durchschnittliche Inzuchtkoeffizient der Flotte beträgt **0.0417**. Liest man die additive Verwandtschaftsmatrix, ist das am wenigsten verwandte Paar der aktuellen Flotte **`G3_T1` & `G3_T3` (Verwandtschaft = 0)** — sie teilen keine gemeinsame Abstammung und bilden die stärkste redundante (N-1-)Paarung, was besonders wichtig ist, weil `G3_T1` selbst eines der am stärksten ingezüchteten Designs ist.

## Datenquellen

| Datensatz | Zeilen | Schlüsselvariablen | Beschreibung |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Ein kleines, deterministisches technisches Pedigree von Verteiltransformator-Designs über drei nicht überlappende Design-Generationen (4 Gründungsplattformen, 4 Ableitungen der zweiten Generation, 4 aktuelle Flottenmodelle). Jedes Nicht-Gründungsdesign verzeichnet die beiden unterschiedlichen Vorgängerdesigns (`Parent1`, `Parent2`), von denen es abgeleitet wurde. `Sex` kennzeichnet die federführende Design-Rolle (M = mechanische/Kern-Linie, F = elektrische/Wicklungs-Linie). Das Pedigree ist von Hand festgelegt — nicht zufällig — sodass die Verwandtschaftsstruktur im Voraus bekannt ist und gegen die Ausgabe der Prozedur geprüft werden kann. |

# Quantifizierung des gemeinsamen Design-Erbes einer Leistungstransformator-Flotte

## Warum sich ein Versorgungsunternehmen für „Inzucht" interessiert

Ein Übertragungs- und Verteilnetzbetreiber betreibt Hunderte von Verteiltransformatoren. Um Konstruktionskosten und Qualifizierungsaufwand zu begrenzen, entwerfen Hersteller selten jeden Transformator von Grund auf neu — jedes neue Modell **erbt** Kerngeometrie, Wicklungstopologie, Isoliersysteme und Durchführungsdesigns von einem oder zwei Vorgängermodellen. Über mehrere Beschaffungszyklen entsteht so eine echte *technische Genealogie*: ein Pedigree von Designs.

Dieses gemeinsame Erbe ist ein verborgenes Zuverlässigkeitsrisiko. Wenn zwei Transformatoren, die dieselbe Last schützen (ein redundantes **N-1**-Paar), vom selben Ahnendesign abstammen, ist ein latenter Konstruktionsfehler — eine Resonanzmode, ein Isolationsalterungsmechanismus, ein Durchführungs-Überschlagspfad — mit hoher Wahrscheinlichkeit in **beiden** Einheiten vorhanden. Eine einzige Grundursache kann dann das redundante Paar gleichzeitig ausfallen lassen: ein *Common-Mode-Ausfall*.

**PROC INBREED** wurde entwickelt, um genau diese Art gemeinsamer Abstammung zu quantifizieren. Obwohl ihre Ursprünge in der Tier- und Pflanzenzucht liegen, ist ihre Mathematik — Wrights additive Verwandtschaftsrekursion — domänenunabhängig: Sie misst den Anteil des Design-Erbes, den zwei Individuen über gemeinsame Vorfahren teilen. Wir übertragen das Genetik-Vokabular auf das Anlagen-Engineering:

| INBREED-Konzept | Interpretation für das Versorgungsunternehmen |
|---|---|
| Individuum | Ein Transformator-Design / -Modell |
| Elternteil (Vater/Mutter) | Ein Vorgängerdesign, von dem es abgeleitet wurde |
| Generation (CLASS) | Ein Beschaffungs-/Design-Zyklus |
| Inzuchtkoeffizient *F* | Grad des selbstähnlichen Erbes innerhalb eines Designs |
| Kovarianz-/Verwandtschaftskoeffizient | Gemeinsames Design-Erbe zwischen zwei Designs |
| Am wenigsten verwandtes Paar | Beste redundante Paarung für N-1-Resilienz |

## Schritt 1 — Das Design-Pedigree festlegen

Wir geben `DesignLineage` direkt ein, damit die Verwandtschaftsstruktur eindeutig ist. Es gibt drei **nicht überlappende Design-Generationen**:

- **Generation 1** — vier Gründungsplattform-Designs (`G1_P1`-`G1_P4`) ohne Vorgänger (leere Elternteile).
- **Generation 2** — vier Ableitungs-Designs (`G2_D1`-`G2_D4`), jeweils entwickelt aus zwei **unterschiedlichen** Generation-1-Plattformen. `G2_D1` und `G2_D2` sind *vollständige Geschwister* (beide aus `G1_P1` x `G1_P2`); `G2_D3` und `G2_D4` sind vollständige Geschwister (beide aus `G1_P3` x `G1_P4`).
- **Generation 3** — vier aktuelle Flottenmodelle (`G3_T1`-`G3_T4`). `G3_T1` wird aus dem Geschwisterpaar `G2_D1` x `G2_D2` gebaut, und `G3_T3` aus dem Geschwisterpaar `G2_D3` x `G2_D4`; diese beiden Designs konzentrieren daher das Erbe. `G3_T2` und `G3_T4` kreuzen jeweils die beiden disjunkten Abstammungslinien.

Die Spalten `ID`, `Parent1` und `Parent2` tragen das Pedigree; `Sex` verzeichnet, welche technische Linie das Design führte. Ein kurzer nachfolgender DATA-Schritt leert den Platzhalter `.`, sodass die Gründungsplattformen leere Elternteile haben, wie PROC INBREED es erwartet.

In [1]:
DATEN DesignLineage;
   LÄNGE ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   EINGABE Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
AUSFÜHREN;

/* Gruendungsplattformen haben keine Vorgaenger: Platzhalterpunkte leeren */
DATEN DesignLineage;
   FESTLEGEN DesignLineage;
   WENN Parent1 = '.' DANN Parent1 = ' ';
   WENN Parent2 = '.' DANN Parent2 = ' ';
AUSFÜHREN;

TITEL 'Stammbaum des Transformator-Designs';
PROZEDUR DRUCKEN DATEN=DesignLineage noobs BEZEICHNUNG;
   VAR Generation ID Parent1 Parent2 Sex;
   BEZEICHNUNG Generation="Generation" ID="Kennung" Parent1="Vorgaenger 1" Parent2="Vorgaenger 2" Sex="Rolle";
AUSFÜHREN;

                                          Stammbaum des Transformator-Designs                                           


Generation  Kennung  Vorgaenger 1  Vorgaenger 2  Rolle
----------  -------  ------------  ------------  -----
         1  G1_P1                                M
         1  G1_P2                                M
         1  G1_P3                                M
         1  G1_P4                                F
         2  G2_D1    G1_P1         G1_P2         M
         2  G2_D2    G1_P1         G1_P2         F
         2  G2_D3    G1_P3         G1_P4         F
         2  G2_D4    G1_P3         G1_P4         M
         3  G3_T1    G2_D1         G2_D2         M
         3  G3_T2    G2_D1         G2_D3         F
         3  G3_T3    G2_D3         G2_D4         F
         3  G3_T4    G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Stammbaum des Transformator-Designs.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Schritt 2 — Inzuchtkoeffizienten nach Design-Generation

Wir führen PROC INBREED im **Mehrgenerationen-Modus** aus, indem wir `Generation` in der `CLASS`-Anweisung benennen, was die Pedigree-Zusammenfassung nach Generation ausgibt. Die `VAR`-Anweisung liefert die drei Pedigree-Spalten in der erforderlichen Reihenfolge: Individuum, erster Vorgänger, zweiter Vorgänger.

- Die Option **IND** gibt den Inzuchtkoeffizienten *F* jedes Designs aus — ein Maß dafür, wie konzentriert dessen eigenes Erbe ist. Ein Design, das aus zwei eng verwandten Vorgängern gebaut wird, trägt ein hohes *F*.
- Die Option **MATRIX** gibt die vollständige additive Verwandtschaftsmatrix aus, sodass wir das paarweise Erbe direkt ablesen können.
- Die Option **AVERAGE** fügt den flottenweiten durchschnittlichen Inzuchtkoeffizienten hinzu — eine einzelne Kennzahl zur Design-Diversität.

Werte nahe 0 bedeuten unabhängige Abstammungslinien; *F* steigt, je enger die beiden Vorgänger eines Designs verwandt sind.

In [2]:
TITEL 'Inzuchtkoeffizienten nach Design-Generation';
PROZEDUR inbreed DATEN=DesignLineage ind average MATRIX;
   VAR ID Parent1 Parent2;
   KLASSE Generation;
   BEZEICHNUNG ID="Kennung" Parent1="Vorgaenger 1" Parent2="Vorgaenger 2" Generation="Generation";
AUSFÜHREN;

                                      Inzuchtkoeffizienten nach Design-Generation                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
Kennung             F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
Kennung             F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Inzuchtkoeffizienten nach Design-Generation.
NOTE: PROC INBREED data=DesignLineage



## Schritt 3 — Kovarianzkoeffizienten für nachgelagertes Risiko-Scoring gespeichert

Ersetzt man die Inzuchtsicht durch die Option **COVAR**, werden dieselben Beziehungen als **Kovarianz- (additive Verwandtschafts-) Koeffizienten** ausgegeben — die Form, die ein Asset-Management-Team in einen Redundanz-Risiko-Score einfließen lassen würde. Die Option **OUTCOV=** erfasst die vollständige Matrix als Datensatz (`DesignCovar`), in dem die Spalten `Col1`-`Col12` die Verwandtschaft jedes Designs zu jedem anderen Design (in Pedigree-Reihenfolge) enthalten — bereit, mit Umspannwerk-Zuordnungen verknüpft zu werden.

Wir drucken den Ausgabedatensatz, damit die gespeicherte Matrix zusammen mit der Liste sichtbar ist.

In [3]:
TITEL 'Kovarianz- (Verwandtschafts-) Koeffizienten';
PROZEDUR inbreed DATEN=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VAR ID Parent1 Parent2;
   KLASSE Generation;
   BEZEICHNUNG ID="Kennung" Parent1="Vorgaenger 1" Parent2="Vorgaenger 2" Generation="Generation";
AUSFÜHREN;

TITEL 'Mit OUTCOV= gespeicherte Verwandtschaftsmatrix fuer die Risikobewertung';
PROZEDUR DRUCKEN DATEN=DesignCovar noobs BEZEICHNUNG;
   BEZEICHNUNG ID="Kennung" Parent1="Vorgaenger 1" Parent2="Vorgaenger 2" Sex="Rolle" Generation="Generation";
AUSFÜHREN;

                                      Kovarianz- (Verwandtschafts-) Koeffizienten                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
Kennung             F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
Kennung                G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Kovarianz- (Verwandtschafts-) Koeffizienten.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Mit OUTCOV= gespeicherte Verwandtschaftsmatrix fuer die Risikobewertung.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Schritt 4 — Am wenigsten verwandte Paarungen für redundante (N-1-)Installationen

Die gespeicherte `DesignCovar`-Matrix ist genau das, was eine Redundanzstudie braucht. Die Verwandtschaft von Design *i* zu Design *j* steht in Zeile *i*, Spalte `Col`*j* (die Spalten sind in Pedigree-Reihenfolge, sodass `Col9`-`Col12` den vier aktuellen Flottenmodellen `G3_T1`-`G3_T4` entsprechen).

Wir lesen die vier Zeilen der aktuellen Flotte aus `DesignCovar`, bilden jedes ungeordnete Paar der aktuellen Flotte, ordnen ihm seinen Verwandtschaftskoeffizienten zu und sortieren nach der geringsten Verwandtschaft zuerst. Ziel ist es, redundante Paare mit dem **niedrigsten** gemeinsamen Erbe zu wählen — diese minimieren die Wahrscheinlichkeit, dass ein vererbter Konstruktionsfehler beide Hälften einer N-1-geschützten Position außer Betrieb setzt.

In [4]:
/* Die vier Zeilen der aktuellen Flotte auslesen; Col9..Col12 sind die
   Verwandtschaften zu G3_T1..G3_T4 (Pedigree-Spaltenreihenfolge).
   Jedes ungeordnete Paar bilden. */
DATEN Pairs;
   FESTLEGEN DesignCovar;
   WO ID in ('G3_T1','G3_T2','G3_T3','G3_T4');
   LÄNGE DesignA $12 DesignB $12;
   FELD g3{4} Col9 Col10 Col11 Col12;
   FELD nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   AUSFÜHRUNG j = 1 BIS 4;
      DesignB = nm{j};
      WENN DesignA < DesignB DANN AUSFÜHRUNG;
         Relationship = g3{j};
         AUSGABE;
      ENDE;
   ENDE;
   BEHALTEN DesignA DesignB Relationship;
AUSFÜHREN;

PROZEDUR SORTIEREN DATEN=Pairs; NACH Relationship; AUSFÜHREN;

TITEL 'Kandidaten fuer redundante (N-1-)Paarungen, am wenigsten verwandt zuerst';
PROZEDUR DRUCKEN DATEN=Pairs noobs BEZEICHNUNG;
   VAR DesignA DesignB Relationship;
   BEZEICHNUNG DesignA="Design A" DesignB="Design B" Relationship="Verwandtschaftsgrad";
AUSFÜHREN;
TITEL;

                        Kandidaten fuer redundante (N-1-)Paarungen, am wenigsten verwandt zuerst                        


Design A  Design B  Verwandtschaftsgrad
--------  --------  -------------------
G3_T1     G3_T3                       0
G3_T2     G3_T4                    0.25
G3_T1     G3_T2                   0.375
G3_T1     G3_T4                   0.375
G3_T2     G3_T3                   0.375
G3_T3     G3_T4                   0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandidaten fuer redundante (N-1-)Paarungen, am wenigsten verwandt zuerst.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretation der Ergebnisse

**Inzuchtkoeffizienten (Schritt 2).** Alle Gründungsplattformen der Generation 1 und alle Ableitungen der Generation 2 zeigen *F* = 0 — konstruktionsbedingt hat keines von ihnen zwei verwandte Vorgänger. Zwei Modelle der Generation 3 weisen *F* = 0.25 auf: `G3_T1` (gebaut aus dem Geschwisterpaar `G2_D1` x `G2_D2`) und `G3_T3` (aus dem Geschwisterpaar `G2_D3` x `G2_D4`). Ihre Vorgänger gehen auf *dasselbe* Paar von Gründungsplattformen zurück, sodass ihr Erbe konzentriert ist. Aus Zuverlässigkeitssicht sind dies die Designs, die am stärksten einem einzelnen vererbten Defekt ausgesetzt sind, und sie rechtfertigen zusätzliche unabhängige Qualifizierungstests. `G3_T2` und `G3_T4`, die die beiden disjunkten Abstammungslinien kreuzen, liegen bei *F* = 0.

**Verwandtschaftsmatrix (Schritt 3).** Die Werte außerhalb der Diagonale quantifizieren das paarweise gemeinsame Erbe. Die beiden Geschwisterpaare der Generation 2 zeigen jeweils eine Verwandtschaft von 0.50 zueinander (`G2_D1`-`G2_D2` und `G2_D3`-`G2_D4`), während Ableitungen aus entgegengesetzten Abstammungslinien bei 0.00 liegen. Die ingezüchteten aktuellen Flottendesigns tragen Selbstverwandtschaften von 1.25 (= 1 + *F*), sichtbar auf der Diagonale. Der Datensatz `DesignCovar` macht die vollständige Matrix verfügbar, um sie mit Umspannwerk-Zuordnungen zu verknüpfen.

**Am wenigsten verwandte Paarungen (Schritt 4).** Ordnet man jedes Paar der aktuellen Flotte nach Verwandtschaft, steht **`G3_T1` & `G3_T3` an erster Stelle bei Verwandtschaft 0.00** — sie stammen von völlig disjunkten Ahnenplattformen ab und teilen kein Design-Erbe. Dies ist die stärkste redundante Paarung, und sie ist besonders wertvoll, weil `G3_T1` selbst eines der beiden am stärksten ingezüchteten Designs ist: Es mit einem völlig unverwandten Partner zu paaren, sichert sein konzentriertes Erbe ab. Das nächstbeste Paar ist `G3_T2` & `G3_T4` bei 0.25; die übrigen Paare liegen bei 0.375. Der durchschnittliche Inzuchtkoeffizient der Flotte von **0.0417** (von der Option AVERAGE in Schritt 2 ausgegeben) fasst die gesamte Design-Diversität zusammen. Die Beschaffung sollte die Paare mit der geringsten Verwandtschaft für N-1-Positionen bevorzugen und die Ahnenbasis im Laufe der Zeit verbreitern — das Anlagen-Engineering-Äquivalent zur Vermeidung eines genetischen Flaschenhalses.

**Vorbehalt.** Dies sind illustrative synthetische Daten; eine Produktionsstudie würde das Pedigree aus den echten Design-Revisionsunterlagen des Herstellers aufbauen und die Verwandtschaftswerte anhand historischer Common-Mode-Ausfallereignisse validieren, bevor sie Beschaffungsentscheidungen leiten.